In [ ]:
-- =============================================================================
-- SLEEPER CORE PIPELINE
-- Dynasty-Ready Analytics Foundation
-- 
-- Key Features:
-- - Player dimension populated from Sleeper players API
-- - Weekly player performance with position and starter status
-- - League configuration for dynasty/keeper/redraft classification
-- - Complete player ownership lifecycle tracking
-- - Enhanced waiver and draft analytics
-- =============================================================================

In [ ]:
-- ---------- DIMENSIONS ----------


In [ ]:
-- League clustering for multi-season tracking
CREATE OR REPLACE MATERIALIZED VIEW dim_league_clusters AS
SELECT
  lower(regexp_replace(coalesce(name, 'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key,
  any_value(name) AS cluster_name,
  min(season) AS season_start,
  max(season) AS season_end
FROM workspace.sleeper_raw.sleeper_league_info_snapshot
GROUP BY lower(regexp_replace(coalesce(name, 'unknown'), '[^a-zA-Z0-9]+', '_'));

In [ ]:
-- League configuration for dynasty/keeper/redraft classification
CREATE OR REPLACE MATERIALIZED VIEW dim_league_config AS
WITH league_base AS (
  SELECT 
    league_id,
    any_value(name) AS league_name,
    min(season) AS season_start,
    max(season) AS season_end,
    any_value(settings) AS settings
  FROM workspace.sleeper_raw.sleeper_league_info_snapshot
  GROUP BY league_id
)
SELECT
  league_id,
  league_name,
  season_start,
  season_end,
  -- Classify league type based on name patterns
  CASE 
    WHEN lower(league_name) LIKE '%inches%' THEN 'dynasty'
    WHEN lower(league_name) LIKE '%empire%' THEN 'keeper'
    ELSE 'redraft'
  END AS league_type,
  -- Extract keeper count if applicable
  CASE 
    WHEN lower(league_name) LIKE '%empire%' THEN 3
    ELSE NULL
  END AS num_keepers,
  CAST(settings['waiver_budget'] AS INT) AS starting_faab,
  CAST(settings['num_teams'] AS INT) AS num_teams,
  lower(regexp_replace(coalesce(league_name, 'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key
FROM league_base;

In [ ]:
-- Manager to roster mapping
CREATE OR REPLACE MATERIALIZED VIEW dim_manager_roster_map AS
WITH ro AS (
  SELECT league_id, roster_id, owner_id
  FROM workspace.sleeper_raw.sleeper_rosters_snapshot
),
us AS (
  SELECT league_id AS u_league_id, user_id AS manager_user_id, display_name
  FROM workspace.sleeper_raw.sleeper_users_snapshot
),
li AS (
  SELECT league_id, season, name 
  FROM workspace.sleeper_raw.sleeper_league_info_snapshot
)
SELECT
  ro.league_id,
  li.season,
  ro.roster_id,
  us.manager_user_id,
  coalesce(us.display_name, 'Unknown') AS manager_display_name,
  lower(regexp_replace(coalesce(li.name,'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key,
  li.name AS cluster_name
FROM ro
LEFT JOIN us ON ro.league_id = us.u_league_id AND ro.owner_id = us.manager_user_id
LEFT JOIN li ON ro.league_id = li.league_id;

In [ ]:
-- Players dimension with data from Sleeper players API
CREATE OR REPLACE MATERIALIZED VIEW dim_players AS
WITH all_player_ids AS (
  -- Get all player IDs that appear in matchups
  SELECT DISTINCT explode(transform(map_keys(players_points), k -> k)) AS player_id
  FROM workspace.sleeper_raw.sleeper_matchups_snapshot
  UNION
  -- Get all player IDs from drafts
  SELECT DISTINCT player_id
  FROM workspace.sleeper_raw.sleeper_draft_picks_snapshot
  WHERE player_id IS NOT NULL
),
players_with_data AS (
  SELECT 
    player_id,
    first_name,
    last_name,
    COALESCE(full_name, CONCAT_WS(' ', first_name, last_name)) AS full_name,
    position,
    team,
    status,
    CAST(age AS INT) AS age,
    CAST(years_exp AS INT) AS years_exp
  FROM workspace.sleeper_raw.players
)
SELECT 
  a.player_id,
  p.full_name,
  p.first_name,
  p.last_name,
  p.position,
  p.team,
  p.status,
  p.age,
  p.years_exp,
  current_timestamp() AS updated_at
FROM all_player_ids a
LEFT JOIN players_with_data p ON a.player_id = p.player_id;

In [ ]:
-- Player ownership lifecycle tracking
-- Critical for dynasty analysis - tracks when players join/leave rosters
CREATE OR REPLACE MATERIALIZED VIEW dim_player_ownership AS
WITH 
-- Step 1: Extract all acquisition events
draft_acquisitions AS (
  SELECT
    dp.player_id,
    dp.league_id,
    dr.season,
    dp.roster_id,
    CAST(NULL AS TIMESTAMP) AS acquired_date,
    1 AS acquired_week,
    CASE 
      WHEN dr.season = (
        SELECT min(season) 
        FROM workspace.sleeper_raw.sleeper_league_info_snapshot 
        WHERE league_id = dp.league_id
      )
      THEN 'startup_draft'
      ELSE 'rookie_draft'
    END AS acquired_via,
    dp.draft_id AS acquired_transaction_id,
    0 AS acquired_faab
  FROM workspace.sleeper_raw.sleeper_draft_picks_snapshot dp
  JOIN workspace.sleeper_raw.sleeper_drafts_snapshot dr ON dp.draft_id = dr.draft_id
  WHERE dp.player_id IS NOT NULL
),
trade_acquisitions_raw AS (
  SELECT
    t.league_id,
    t.transaction_id,
    t.leg,
    t.created,
    li.season,
    explode(transform(map_entries(t.adds), e -> 
      named_struct('player_id', e.key, 'to_roster_id', e.value)
    )) AS add_struct
  FROM workspace.sleeper_raw.sleeper_transactions_snapshot t
  JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li USING (league_id)
  WHERE t.type = 'trade' AND t.status = 'complete'
),
trade_acquisitions AS (
  SELECT
    add_struct.player_id,
    league_id,
    season,
    add_struct.to_roster_id AS roster_id,
    to_timestamp(created/1000.0) AS acquired_date,
    leg AS acquired_week,
    'trade' AS acquired_via,
    transaction_id AS acquired_transaction_id,
    0 AS acquired_faab
  FROM trade_acquisitions_raw
),
waiver_acquisitions_raw AS (
  SELECT
    t.league_id,
    t.transaction_id,
    t.leg,
    t.created,
    t.type,
    t.waiver_bid,
    li.season,
    explode(transform(map_entries(t.adds), e -> 
      named_struct('player_id', e.key, 'to_roster_id', e.value)
    )) AS add_struct
  FROM workspace.sleeper_raw.sleeper_transactions_snapshot t
  JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li USING (league_id)
  WHERE t.type IN ('waiver', 'free_agent')
),
waiver_acquisitions AS (
  SELECT
    add_struct.player_id,
    league_id,
    season,
    add_struct.to_roster_id AS roster_id,
    to_timestamp(created/1000.0) AS acquired_date,
    leg AS acquired_week,
    type AS acquired_via,
    transaction_id AS acquired_transaction_id,
    coalesce(waiver_bid, 0) AS acquired_faab
  FROM waiver_acquisitions_raw
),
-- Step 2: Extract all departure events
trade_departures_raw AS (
  SELECT
    t.league_id,
    t.transaction_id,
    t.leg,
    t.created,
    li.season,
    explode(transform(map_entries(t.drops), e -> 
      named_struct('player_id', e.key, 'from_roster_id', e.value)
    )) AS drop_struct
  FROM workspace.sleeper_raw.sleeper_transactions_snapshot t
  JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li USING (league_id)
  WHERE t.type = 'trade' AND t.status = 'complete'
),
trade_departures AS (
  SELECT
    drop_struct.player_id,
    league_id,
    season,
    drop_struct.from_roster_id AS roster_id,
    to_timestamp(created/1000.0) AS departed_date,
    leg AS departed_week,
    'trade' AS departed_via,
    transaction_id AS departed_transaction_id
  FROM trade_departures_raw
),
drop_departures_raw AS (
  SELECT
    t.league_id,
    t.transaction_id,
    t.leg,
    t.created,
    li.season,
    explode(transform(map_entries(t.drops), e -> 
      named_struct('player_id', e.key, 'from_roster_id', e.value)
    )) AS drop_struct
  FROM workspace.sleeper_raw.sleeper_transactions_snapshot t
  JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li USING (league_id)
  WHERE t.type IN ('waiver', 'free_agent')
),
drop_departures AS (
  SELECT
    drop_struct.player_id,
    league_id,
    season,
    drop_struct.from_roster_id AS roster_id,
    to_timestamp(created/1000.0) AS departed_date,
    leg AS departed_week,
    'drop' AS departed_via,
    transaction_id AS departed_transaction_id
  FROM drop_departures_raw
),
-- Step 3: Combine all events
all_acquisitions AS (
  SELECT * FROM draft_acquisitions
  UNION ALL
  SELECT * FROM trade_acquisitions
  UNION ALL  
  SELECT * FROM waiver_acquisitions
),
all_departures AS (
  SELECT * FROM trade_departures
  UNION ALL
  SELECT * FROM drop_departures
),
-- Step 4: Match acquisitions to their next departure
ownership_periods AS (
  SELECT
    a.player_id,
    a.league_id,
    a.season,
    a.roster_id,
    a.acquired_date,
    a.acquired_week,
    a.acquired_via,
    a.acquired_transaction_id,
    a.acquired_faab,
    d.departed_date,
    d.departed_week,
    d.departed_via,
    d.departed_transaction_id,
    ROW_NUMBER() OVER (
      PARTITION BY a.player_id, a.league_id, a.roster_id, a.season, a.acquired_transaction_id
      ORDER BY coalesce(d.departed_date, CAST('2099-01-01' AS TIMESTAMP)) ASC
    ) AS rn
  FROM all_acquisitions a
  LEFT JOIN all_departures d
    ON a.player_id = d.player_id
    AND a.league_id = d.league_id  
    AND a.roster_id = d.roster_id
    AND a.season = d.season
    AND coalesce(d.departed_date, CAST('2099-01-01' AS TIMESTAMP)) > 
        coalesce(a.acquired_date, CAST('1900-01-01' AS TIMESTAMP))
)
SELECT
  md5(concat(player_id, league_id, CAST(roster_id AS STRING), acquired_transaction_id)) AS ownership_id,
  player_id,
  league_id,
  season,
  roster_id,
  acquired_date,
  acquired_week,
  acquired_via,
  acquired_transaction_id,
  acquired_faab,
  departed_date,
  departed_week,
  departed_via,
  departed_transaction_id,
  CASE WHEN departed_date IS NULL THEN TRUE ELSE FALSE END AS is_current_roster,
  CASE 
    WHEN departed_week IS NOT NULL AND acquired_week IS NOT NULL
    THEN departed_week - acquired_week
    ELSE NULL
  END AS weeks_owned
FROM ownership_periods
WHERE rn = 1;

In [ ]:
-- ---------- FACTS ----------


In [ ]:
-- Team performance by week
CREATE OR REPLACE MATERIALIZED VIEW fact_team_week AS
WITH base AS (
  SELECT m.league_id, m.week, m.matchup_id, m.roster_id, m.points AS points_for
  FROM workspace.sleeper_raw.sleeper_matchups_snapshot m
),
opp AS (
  SELECT a.league_id, a.week, a.matchup_id, a.roster_id,
         a.points_for, b.points_for AS points_against
  FROM base a
  LEFT JOIN base b
    ON a.league_id=b.league_id AND a.week=b.week
   AND a.matchup_id=b.matchup_id AND a.roster_id<>b.roster_id
),
li AS (
  SELECT league_id, season FROM workspace.sleeper_raw.sleeper_league_info_snapshot
),
wmed AS (
  SELECT o.league_id, l.season, o.week,
         percentile_approx(o.points_for, 0.5) AS med
  FROM opp o JOIN li l USING (league_id)
  GROUP BY o.league_id, l.season, o.week
)
SELECT o.league_id, l.season, o.week, o.roster_id,
       o.points_for, o.points_against,
       (o.points_for >= w.med) AS median_beat_flag
FROM opp o
JOIN li l ON o.league_id = l.league_id
JOIN wmed w ON o.league_id=w.league_id AND l.season=w.season AND o.week=w.week;

In [ ]:
-- Player performance by week with position and starter status
CREATE OR REPLACE MATERIALIZED VIEW fact_player_week AS
WITH src AS (
  SELECT m.league_id, li.season, m.week, m.roster_id,
         m.starters,
         map_entries(m.players_points) AS entries
  FROM workspace.sleeper_raw.sleeper_matchups_snapshot m
  JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li USING (league_id)
),
rows AS (
  SELECT league_id, season, week, roster_id, starters,
         transform(entries, e -> named_struct('player_id', e.key, 'points', e.value)) AS rows
  FROM src
),
exploded AS (
  SELECT league_id, season, week, roster_id, starters, explode(rows) AS r
  FROM rows
)
SELECT 
  e.league_id, 
  e.season, 
  e.week, 
  e.roster_id,
  e.r.player_id, 
  e.r.points,
  CAST(NULL AS DOUBLE) AS projected_points,
  p.position,
  array_contains(e.starters, e.r.player_id) AS was_started,
  current_timestamp() AS updated_at
FROM exploded e
LEFT JOIN dim_players p ON e.r.player_id = p.player_id;

In [ ]:
-- Enriched player week with cluster info
CREATE OR REPLACE MATERIALIZED VIEW fact_player_week_enriched AS
SELECT f.*,
       lower(regexp_replace(coalesce(li.name,'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key,
       li.name AS cluster_name
FROM fact_player_week f
JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li USING (league_id);


In [ ]:
-- Standings by week
CREATE OR REPLACE MATERIALIZED VIEW fact_standings_week AS
WITH wl AS (
  SELECT league_id, season, week, roster_id,
         (points_for > points_against)  AS win_flag,
         (points_for < points_against)  AS loss_flag,
         (points_for = points_against)  AS tie_flag,
         median_beat_flag,
         points_for, points_against
  FROM fact_team_week
)
SELECT league_id, season, week, roster_id,
       SUM(CASE WHEN win_flag  THEN 1 ELSE 0 END)
         OVER (PARTITION BY league_id, season, roster_id ORDER BY week
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS wins,
       SUM(CASE WHEN loss_flag THEN 1 ELSE 0 END)
         OVER (PARTITION BY league_id, season, roster_id ORDER BY week
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS losses,
       SUM(CASE WHEN tie_flag  THEN 1 ELSE 0 END)
         OVER (PARTITION BY league_id, season, roster_id ORDER BY week
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS ties,
       points_for, points_against,
       SUM(CASE WHEN median_beat_flag THEN 1 ELSE 0 END)
         OVER (PARTITION BY league_id, season, roster_id ORDER BY week
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS expected_wins
FROM wl;

In [ ]:
-- Waiver acquisitions fact table
CREATE OR REPLACE MATERIALIZED VIEW fact_waiver_acquisitions AS
WITH waiver_adds AS (
  SELECT
    t.transaction_id,
    t.league_id,
    t.leg AS week,
    coalesce(t.waiver_bid, 0) AS faab_bid,
    CASE WHEN t.type = 'waiver' THEN TRUE ELSE FALSE END AS was_waiver,
    to_timestamp(t.created/1000.0) AS acquired_date,
    explode(transform(map_entries(t.adds), e -> 
      named_struct('player_id', e.key, 'to_roster_id', e.value)
    )) AS a
  FROM workspace.sleeper_raw.sleeper_transactions_snapshot t
  WHERE t.type IN ('waiver', 'free_agent')
)
SELECT
  wa.transaction_id,
  wa.league_id,
  li.season,
  wa.week,
  wa.a.to_roster_id AS roster_id,
  wa.a.player_id,
  wa.faab_bid,
  wa.was_waiver,
  wa.acquired_date
FROM waiver_adds wa
JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li USING (league_id);

In [ ]:
-- ---------- AGGREGATES ----------


In [ ]:
-- Team consistency metrics (unchanged)
CREATE OR REPLACE MATERIALIZED VIEW agg_consistency_metrics AS
SELECT league_id, season, roster_id,
       stddev_pop(points_for) AS points_for_stddev,
       avg(points_for)       AS points_for_avg,
       count(*)              AS games_played,
       CASE WHEN avg(points_for) <> 0 THEN stddev_pop(points_for)/avg(points_for) END AS points_for_cv
FROM fact_team_week
GROUP BY league_id, season, roster_id;


In [ ]:
-- Head to head records (unchanged)
CREATE OR REPLACE MATERIALIZED VIEW agg_rivalry_head_to_head AS
WITH a AS (SELECT * FROM fact_team_week),
     b AS (SELECT * FROM fact_team_week)
SELECT a.league_id, a.season,
       a.roster_id AS roster_id_a, b.roster_id AS roster_id_b,
       count(*) AS games_played,
       sum(CASE WHEN a.points_for > b.points_for THEN 1 ELSE 0 END) AS wins_a,
       sum(CASE WHEN a.points_for < b.points_for THEN 1 ELSE 0 END) AS wins_b,
       sum(CASE WHEN a.points_for = b.points_for THEN 1 ELSE 0 END) AS ties,
       avg(a.points_for) AS pf_per_game_a,
       avg(b.points_for) AS pf_per_game_b
FROM a JOIN b
  ON a.league_id=b.league_id AND a.season=b.season AND a.week=b.week
 AND a.roster_id<>b.roster_id
GROUP BY a.league_id, a.season, a.roster_id, b.roster_id;


In [ ]:
-- All-time records (unchanged)
CREATE OR REPLACE MATERIALIZED VIEW agg_records_all_time AS
SELECT league_id, roster_id,
       max(points_for) AS max_points_for,
       min(points_for) AS min_points_for
FROM fact_team_week
GROUP BY league_id, roster_id;


In [ ]:
-- Player total points per roster (answers "most points scored for team")
CREATE OR REPLACE MATERIALIZED VIEW agg_player_roster_totals AS
SELECT 
  pw.league_id,
  pw.roster_id,
  pw.player_id,
  p.full_name,
  p.position,
  min(pw.season) AS first_season,
  max(pw.season) AS last_season,
  count(DISTINCT pw.season) AS seasons_count,
  count(*) AS weeks_count,
  sum(pw.points) AS total_points,
  avg(pw.points) AS points_per_week,
  sum(CASE WHEN pw.was_started THEN pw.points ELSE 0 END) AS points_as_starter
FROM fact_player_week pw
LEFT JOIN dim_players p ON pw.player_id = p.player_id
GROUP BY pw.league_id, pw.roster_id, pw.player_id, p.full_name, p.position;

In [ ]:
-- Draft ROI by round (enhanced with career tracking)
CREATE OR REPLACE MATERIALIZED VIEW agg_draft_roi_by_round AS
WITH picks AS (
  SELECT
    p.league_id,
    li.season,
    CAST(p.round AS INT) AS round,
    p.player_id
  FROM workspace.sleeper_raw.sleeper_draft_picks_snapshot p
  JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li USING (league_id)
),
player_season_points AS (
  SELECT
    league_id, season, player_id,
    SUM(points) AS season_points
  FROM fact_player_week
  GROUP BY league_id, season, player_id
),
player_career_points AS (
  SELECT
    league_id, player_id,
    SUM(points) AS career_points
  FROM fact_player_week
  GROUP BY league_id, player_id
),
rook AS (
  SELECT
    pk.league_id, pk.season, pk.round, pk.player_id,
    COALESCE(psp.season_points, 0.0) AS rookie_season_points,
    COALESCE(pcp.career_points, 0.0) AS career_points
  FROM picks pk
  LEFT JOIN player_season_points psp
    ON pk.league_id = psp.league_id
   AND pk.season    = psp.season
   AND pk.player_id = psp.player_id
  LEFT JOIN player_career_points pcp
    ON pk.league_id = pcp.league_id
   AND pk.player_id = pcp.player_id
),
by_round AS (
  SELECT
    league_id, season, round,
    AVG(rookie_season_points) AS avg_rookie_points,
    AVG(career_points) AS avg_career_points,
    COUNT(*) AS picks_count
  FROM rook
  GROUP BY league_id, season, round
),
season_avg AS (
  SELECT
    league_id, season,
    AVG(rookie_season_points) AS season_avg_points,
    AVG(career_points) AS career_avg_points
  FROM rook
  GROUP BY league_id, season
)
SELECT
  b.league_id,
  b.season,
  b.round,
  b.picks_count,
  b.avg_rookie_points,
  b.avg_career_points,
  s.season_avg_points,
  s.career_avg_points,
  CASE WHEN s.season_avg_points > 0
       THEN b.avg_rookie_points / s.season_avg_points
       ELSE NULL
  END AS rookie_roi,
  CASE WHEN s.career_avg_points > 0
       THEN b.avg_career_points / s.career_avg_points
       ELSE NULL
  END AS career_roi
FROM by_round b
JOIN season_avg s
  ON b.league_id = s.league_id AND b.season = s.season;
